In [ ]:
# Colab setup - run this cell first if you're on Google Colab
try:
    import google.colab
    print("Colab environment ready! All required packages are pre-installed.")
except ImportError:
    pass  # Not on Colab, no action needed

In [ ]:
# Core PyTorch
import torch
from torch import tensor
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

# Vision
from torchvision import datasets, transforms

# Standard libraries
from pathlib import Path
from PIL import Image
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

matplotlib.rc('image', cmap='Greys')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
# === Plotting helpers (called from cells below, keeps logic cells clean) ===

def plot_conv_trace(patch, kernel, title="Convolution Trace", figsize=(16, 4)):
    """Visualize a convolution step: patch × kernel = products → sum.
    Shows 4 grids side by side with numbers in each cell."""
    products = patch.float() * kernel
    total = products.sum().item()

    fig, axes = plt.subplots(1, 4, figsize=figsize)
    
    data_list = [patch.float().numpy(), kernel.numpy(), products.numpy()]
    titles = ["Image Patch", "Kernel", "Element-wise Products"]
    cmaps = ["gray", "RdBu_r", "RdBu_r"]
    
    for ax, data, t, cmap in zip(axes[:3], data_list, titles, cmaps):
        h, w = data.shape
        vmax = max(abs(data.max()), abs(data.min()), 1)
        if cmap == "gray":
            ax.imshow(data, cmap=cmap, vmin=0, vmax=255)
        else:
            ax.imshow(data, cmap=cmap, vmin=-vmax, vmax=vmax)
        for i in range(h):
            for j in range(w):
                val = data[i, j]
                color = "white" if (cmap == "gray" and val < 128) else "black"
                if cmap != "gray":
                    color = "white" if abs(val) > vmax * 0.6 else "black"
                ax.text(j, i, f"{val:.0f}", ha="center", va="center",
                       fontsize=13, fontweight="bold", color=color)
        ax.set_title(t, fontsize=12)
        ax.set_xticks([])
        ax.set_yticks([])
    
    # Fourth panel: the sum result
    ax = axes[3]
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    color = "#2ecc71" if total > 50 else "#e74c3c" if total < -50 else "#95a5a6"
    ax.add_patch(plt.Rectangle((0.1, 0.2), 0.8, 0.6, facecolor=color, alpha=0.3, edgecolor=color, linewidth=2))
    ax.text(0.5, 0.5, f"Sum\n{total:.0f}", ha="center", va="center",
           fontsize=22, fontweight="bold", color=color)
    sign_label = "(strong match!)" if total > 50 else "(opposite!)" if total < -50 else "(no match)"
    ax.text(0.5, 0.15, sign_label, ha="center", va="center", fontsize=11, color=color)
    ax.set_title("→ Result", fontsize=12)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.axis("off")
    
    fig.suptitle(title, fontsize=14, fontweight="bold", y=1.02)
    plt.tight_layout()
    plt.show()


def plot_kernel_comparison(patch, kernels, kernel_names, title="Same Patch, Different Kernels"):
    """Show one patch tested against multiple kernels, with results compared."""
    n = len(kernels)
    fig, axes = plt.subplots(1, n + 1, figsize=(5 * (n + 1), 4))
    
    # Show the patch
    ax = axes[0]
    data = patch.float().numpy()
    ax.imshow(data, cmap="gray", vmin=0, vmax=255)
    h, w = data.shape
    for i in range(h):
        for j in range(w):
            val = data[i, j]
            color = "white" if val < 128 else "black"
            ax.text(j, i, f"{val:.0f}", ha="center", va="center",
                   fontsize=13, fontweight="bold", color=color)
    ax.set_title("Image Patch", fontsize=12)
    ax.set_xticks([])
    ax.set_yticks([])
    
    # Show each kernel with its result
    for idx, (kernel, name) in enumerate(zip(kernels, kernel_names)):
        ax = axes[idx + 1]
        kdata = kernel.numpy()
        products = (patch.float() * kernel).sum().item()
        vmax = max(abs(kdata.max()), abs(kdata.min()), 1)
        ax.imshow(kdata, cmap="RdBu_r", vmin=-vmax, vmax=vmax)
        for i in range(h):
            for j in range(w):
                val = kdata[i, j]
                color = "white" if abs(val) > vmax * 0.6 else "black"
                ax.text(j, i, f"{val:.0f}", ha="center", va="center",
                       fontsize=13, fontweight="bold", color=color)
        result_color = "#2ecc71" if products > 50 else "#e74c3c" if products < -50 else "#95a5a6"
        ax.set_title(f"{name}\nResult: {products:.0f}", fontsize=12, color=result_color, fontweight="bold")
        ax.set_xticks([])
        ax.set_yticks([])
    
    fig.suptitle(title, fontsize=14, fontweight="bold", y=1.02)
    plt.tight_layout()
    plt.show()



def plot_activation_drift(rows, title='Activation distributions through layers'):
    """Plot side-by-side activation distribution histograms across layers.
    rows: list of dicts with keys 'values_per_layer', 'label', 'color'.
    Filters out exact zeros (dead neurons after ReLU) and annotates % dead."""
    n_layers = len(rows[0]['values_per_layer'])
    fig, axes = plt.subplots(len(rows), n_layers, figsize=(14, 2.5 * len(rows)))
    if len(rows) == 1:
        axes = [axes]
    for row_idx, row in enumerate(rows):
        for i, (ax, vals) in enumerate(zip(axes[row_idx], row['values_per_layer'])):
            pct_zero = (np.abs(vals) < 1e-7).mean() * 100
            nonzero = vals[np.abs(vals) >= 1e-7]
            if len(nonzero) > 0:
                ax.hist(nonzero, bins=60, color=row['color'], alpha=0.7, density=True)
            ax.set_xlim(-5, 5)
            ax.axvline(0, color='red', linestyle='--', alpha=0.3)
            if pct_zero > 1:
                ax.text(0.95, 0.95, f'{pct_zero:.0f}% dead', transform=ax.transAxes,
                        ha='right', va='top', fontsize=9, color='#cc0000',
                        bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))
            if row_idx == 0:
                ax.set_title(f'Layer {i+1}', fontsize=12)
            if i == 0:
                ax.set_ylabel(row['label'], fontsize=12, fontweight='bold')
    fig.suptitle(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

print("Plotting helpers loaded!")

# === Image display helper ===

def show_image(img, ax=None, cmap='Greys', title=None, **kwargs):
    """Display a tensor or array as an image."""
    if ax is None:
        _, ax = plt.subplots()
    if isinstance(img, torch.Tensor):
        img = img.detach().cpu()
        if img.ndim == 3 and img.shape[0] in (1, 3):
            img = img.permute(1, 2, 0).squeeze()
    ax.imshow(img, cmap=cmap, **kwargs)
    ax.axis('off')
    if title:
        ax.set_title(title)

# === Training loop helpers ===

def train_one_epoch(model, loader, criterion, optimizer, scheduler=None):
    """Train for one epoch, return (loss, accuracy)."""
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        out = model(xb)
        loss = criterion(out, yb)
        loss.backward()
        optimizer.step()
        if scheduler:
            scheduler.step()
        total_loss += loss.item() * xb.size(0)
        correct += (out.argmax(1) == yb).sum().item()
        total += yb.size(0)
    return total_loss / total, correct / total

def evaluate(model, loader, criterion):
    """Evaluate model, return (loss, accuracy)."""
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            out = model(xb)
            loss = criterion(out, yb)
            total_loss += loss.item() * xb.size(0)
            correct += (out.argmax(1) == yb).sum().item()
            total += yb.size(0)
    return total_loss / total, correct / total

# === Activation statistics tracker ===

class ActivationTracker:
    """Track mean, std, and % near-zero activations per layer during training."""
    def __init__(self, model):
        self.stats = {}
        self.hooks = []
        for i, layer in enumerate(model):
            if isinstance(layer, nn.Sequential):
                self.stats[i] = {'mean': [], 'std': [], 'pct_near_zero': []}
                def hook(module, inp, out, idx=i):
                    vals = out.detach().cpu()
                    self.stats[idx]['mean'].append(vals.mean().item())
                    self.stats[idx]['std'].append(vals.std().item())
                    self.stats[idx]['pct_near_zero'].append(
                        (vals.abs() < 0.05).float().mean().item() * 100)
                self.hooks.append(layer.register_forward_hook(hook))

    def remove_hooks(self):
        for h in self.hooks:
            h.remove()

    def plot_layer_stats(self, idx):
        """Plot mean, std, % near-zero for layer at index idx. Negative indices work."""
        keys = sorted(self.stats.keys())
        if idx < 0:
            idx = keys[idx]
        s = self.stats[idx]
        fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(14, 3))
        ax1.plot(s['mean']); ax1.set_title('Mean'); ax1.set_xlabel('Batch')
        ax2.plot(s['std']); ax2.set_title('Std'); ax2.set_xlabel('Batch')
        ax3.plot(s['pct_near_zero']); ax3.set_title('% Near Zero'); ax3.set_xlabel('Batch')
        plt.tight_layout()
        plt.show()


def plot_activation_drift(rows, title='Activation distributions through layers'):
    """Plot side-by-side activation distribution histograms across layers.
    rows: list of dicts with keys 'values_per_layer', 'label', 'color'.
    Filters out exact zeros (dead neurons after ReLU) and annotates % dead."""
    n_layers = len(rows[0]['values_per_layer'])
    fig, axes = plt.subplots(len(rows), n_layers, figsize=(14, 2.5 * len(rows)))
    if len(rows) == 1:
        axes = [axes]
    for row_idx, row in enumerate(rows):
        for i, (ax, vals) in enumerate(zip(axes[row_idx], row['values_per_layer'])):
            pct_zero = (np.abs(vals) < 1e-7).mean() * 100
            nonzero = vals[np.abs(vals) >= 1e-7]
            if len(nonzero) > 0:
                ax.hist(nonzero, bins=60, color=row['color'], alpha=0.7, density=True)
            ax.set_xlim(-5, 5)
            ax.axvline(0, color='red', linestyle='--', alpha=0.3)
            if pct_zero > 1:
                ax.text(0.95, 0.95, f'{pct_zero:.0f}% dead', transform=ax.transAxes,
                        ha='right', va='top', fontsize=9, color='#cc0000',
                        bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))
            if row_idx == 0:
                ax.set_title(f'Layer {i+1}', fontsize=12)
            if i == 0:
                ax.set_ylabel(row['label'], fontsize=12, fontweight='bold')
    fig.suptitle(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

print("Plotting helpers loaded!")

All imports are in the cells above. No external dependencies beyond PyTorch and torchvision.

# Convolutional Neural Networks

In L9 and L10 we built image classifiers using MLPs. We were able to get decent accuracy, but we also saw the limitations: MLPs flatten the image into a vector, destroying all spatial structure. A pixel in the top-left corner is treated the same as one in the center. They also use a huge number of parameters since every pixel connects to every neuron.

In this lesson, we'll dig into what convolutions are and build a CNN from scratch. We'll then study a range of techniques to improve training stability: larger batches, learning rate scheduling, and batch normalization.

## The Magic of Convolutions

One of the most powerful tools that machine learning practitioners have at their disposal is *feature engineering*. A *feature* is a transformation of the data which is designed to make it easier to model. For instance, extracting the day-of-week from a date column in tabular data. What kinds of features might we be able to create from images?

📖 **Jargon: Feature engineering: Creating new transformations of the input data in order to make it easier to model.**



In the context of an image, a feature is a visually distinctive attribute. For example, the number 7 is characterized by a horizontal edge near the top of the digit, and a top-right to bottom-left diagonal edge underneath that. The number 3 has diagonal edges at the top and bottom, horizontal edges at the middle and extremes, and curves connecting them. So what if we could extract information about where the edges occur in each image, and use that as our features, instead of raw pixels?

Finding edges in an image is a very common task in computer vision, and surprisingly straightforward. We use something called a *convolution*. A convolution requires nothing more than multiplication and addition — two operations that are responsible for the vast majority of work in every deep learning model.

A convolution applies a *kernel* across an image. A kernel is a small matrix — the 3×3 matrix in the top right of the diagram below is an example.

<img src="images/chapter9_conv_basic.png" id="basic_conv" caption="Applying a kernel to one location" alt="Applying a kernel to one location" width="700">

The 7×7 grid to the left is the *image* we're going to apply the kernel to. The convolution operation multiplies each element of the kernel by each element of a 3×3 block of the image. The results of these multiplications are then added together. The diagram above shows an example of applying a kernel to a single location in the image, the 3×3 block around cell 18.

Let's do this with code. First, we create a little 3×3 matrix like so:

In [ ]:
# A 3x3 kernel for detecting top edges.
# Row of -1s on top, 0s in middle, 1s on bottom.
# Where pixel values jump from dark (top) to light (bottom) → large positive output.
# Where they jump from light to dark → large negative output.
top_edge = tensor([[-1,-1,-1],
                   [ 0, 0, 0],
                   [ 1, 1, 1]]).float()

This is our *kernel* — the small matrix of weights that defines what pattern we're looking for. We'll also need an image to apply it to:

In [ ]:
# Download MNIST dataset (handwritten digits 0-9, 28x28 grayscale)
mnist_raw = datasets.MNIST(root='./data', train=True, download=True)

# Grab a single "3" image for our manual convolution examples
# mnist_raw.data is a tensor of shape (60000, 28, 28), uint8
idx_3 = (mnist_raw.targets == 3).nonzero(as_tuple=True)[0][5]  # pick the 6th "3"
im3 = mnist_raw.data[idx_3]  # shape: (28, 28), uint8 tensor
path = Path('./data/MNIST')

In [ ]:
# Display the digit "3" as an image
# This is a 28x28 grayscale image, values 0-255
show_image(im3)
plt.title("A handwritten 3")
plt.show()

Now we're going to take the top 3×3-pixel square of our image, and multiply each of those values by each item in our kernel. Then we'll add them up, like so:

In [ ]:
# im3 is already a tensor (28x28, uint8) from torchvision
# Let's rename it for clarity
im3_t = im3

# Take the top-left 3x3 patch and multiply element-wise with the kernel
# Each pixel x corresponding kernel weight
im3_t[0:3,0:3].float() * top_edge

In [ ]:
# Visualize: what happens when we convolve the top-left corner (all zeros)?
patch = im3_t[0:3, 0:3]
plot_conv_trace(patch, top_edge, title="Top-left corner: no edge (all zeros)")

The top-left corner is all zeros (white background) — so every multiplication is `0 × something = 0`. The sum is zero. No edge detected, which makes sense: there's nothing there.

This is the core operation of a convolution: **overlay the kernel on a patch, multiply each pair of numbers, sum the products.** The result is a single number that tells you how strongly that patch matches the pattern the kernel is looking for.

Now let's look at patches where there actually *is* an edge — the numbers will be much more interesting.

In [ ]:
# Sum the element-wise products → one number = the convolution output for this position
# Near zero here because the top-left corner is all white (uniform → no edge)
(im3_t[0:3,0:3] * top_edge).sum()

Not very interesting so far—all the pixels in the top-left corner are white. But let's pick a couple of more interesting spots:

In [ ]:
# Visualize the top-left corner of the image as a heatmap of pixel values
# Darker cells = higher pixel values (closer to the stroke of the digit)
df = pd.DataFrame(im3_t[:10,:20])
df.style.set_properties(**{'font-size':'6pt'}).background_gradient('Greys')

<img alt="Top section of a digit" width="490" src="images/att_00059.png">

There's a top edge at cell 5,8. Let's repeat our calculation there:

In [ ]:
# Apply kernel at row 4-7, col 6-9 — there's a top edge here (dark above, light below)
# Should return a large positive number
(im3_t[4:7,6:9] * top_edge).sum()

In [ ]:
# Visualize: what happens at a REAL top edge (dark above, bright below)?
patch_edge = im3_t[4:7, 6:9]
plot_conv_trace(patch_edge, top_edge, title="Top edge: dark above, bright below")

Look at the products grid. The kernel's `-1` row (top) multiplied by dark pixels (≈0) produces near-zero. The kernel's `+1` row (bottom) multiplied by bright pixels (254) produces large positives. The middle row is always zero. The sum is strongly positive: "yes, there's a top edge here."

When the pattern is reversed (bright above, dark below), the `-1` row hits bright pixels and produces large *negatives*. The sum flips negative: "this is the opposite of a top edge."

There's a right edge at cell 8,18. What does that give us?:

In [ ]:
# Apply at row 7-10, col 17-20 — this is a bottom edge (light above, dark below)
# Should return a negative number (opposite of what top_edge detects)
(im3_t[7:10,17:20] * top_edge).sum()

In [ ]:
# Visualize: bottom edge — the opposite pattern
patch_bottom = im3_t[7:10, 17:20]
plot_conv_trace(patch_bottom, top_edge, title="Bottom edge: bright above, dark below")

As you can see, this calculation returns a high number where the 3×3 patch represents a top edge (low values at the top, high values underneath). The `-1` values in our kernel cancel out the dark pixels, while the `+1` values amplify the bright ones.

Let's look at the math. The kernel slides over every 3×3 window in the image. If we name the pixel values:

$$\begin{matrix} a1 & a2 & a3 \\ a4 & a5 & a6 \\ a7 & a8 & a9 \end{matrix}$$

it computes $-a1-a2-a3+a7+a8+a9$. If the top and bottom rows have similar brightness, the terms cancel and we get 0 — no edge. If the bottom row is brighter than the top, we get a large positive number — a top edge. If the top is brighter, we get a large negative — the opposite edge.

Flipping the kernel (putting `+1` on top and `-1` on the bottom) would detect edges going from dark to light. Putting the `+1`s and `-1`s in columns instead of rows gives us kernels that detect vertical edges. Each set of weights produces a different kind of pattern detector.

Let's create a function to do this for one location, and check it matches our result from before:

In [ ]:
# Helper: apply a 3x3 kernel centered at (row, col) in the image
# Grabs the 3x3 patch around that position, element-wise multiplies with kernel, sums
def apply_kernel(row, col, kernel):
    return (im3_t[row-1:row+2,col-1:col+2] * kernel).sum()

In [ ]:
# Test: same result as our manual calculation above
apply_kernel(5,7,top_edge)

But note that we can't apply it to the corner (e.g., location 0,0), since there isn't a complete 3×3 square there.

### Mapping a Convolution Kernel

We can map `apply_kernel()` across the coordinate grid. That is, we'll be taking our 3×3 kernel, and applying it to each 3×3 section of our image. For instance, the diagram below shows the positions a 3×3 kernel can be applied to in the first row of a 5×5 image.

<img src="images/chapter9_nopadconv.svg" id="nopad_conv" caption="Applying a kernel across a grid" alt="Applying a kernel across a grid" width="400">

To get a grid of coordinates we can use a *nested list comprehension*, like so:

In [ ]:
# Nested list comprehension: generates a grid of (row, col) coordinates
# This is the pattern we'll use to apply the kernel at every valid position
[[(i,j) for j in range(1,5)] for i in range(1,5)]

📝 **Note:** Nested List Comprehensions: Nested list comprehensions are used a lot in Python, so if you haven't seen them before, take a few minutes to make sure you understand what's happening here, and experiment with writing your own nested list comprehensions.

Here's the result of applying our kernel over a coordinate grid:

In [ ]:
# Apply the top-edge kernel at every valid position in the 28x28 image
# range(1,27) because we can't center a 3x3 kernel on the border pixels
rng = range(1,27)
top_edge3 = tensor([[apply_kernel(i,j,top_edge) for j in rng] for i in rng])

# Bright pixels = strong top edges detected, dark = bottom edges
show_image(top_edge3)
plt.show()

Looking good! Our top edges are black, and bottom edges are white (since they are the *opposite* of top edges). Now that our image contains negative numbers too, `matplotlib` has automatically changed our colors so that white is the smallest number in the image, black the highest, and zeros appear as gray.

We can try the same thing for left edges:

In [ ]:
# Left-edge kernel: detects vertical edges (dark on left, light on right)
# -1s in left column, 1s in middle column, 0s on right
left_edge = tensor([[-1,1,0],
                    [-1,1,0],
                    [-1,1,0]]).float()

# Apply across the whole image, same as before
left_edge3 = tensor([[apply_kernel(i,j,left_edge) for j in rng] for i in rng])

show_image(left_edge3)
plt.show()

In [ ]:
# Same patch tested with two different kernels — which edge does it detect?
patch_compare = im3_t[4:7, 6:9]
plot_kernel_comparison(
    patch_compare,
    [top_edge, left_edge],
    ["Top-edge kernel", "Left-edge kernel"],
    title="Same patch, different kernels → different responses"
)

The top-edge kernel fires strongly (762) because there's a clear horizontal edge. The left-edge kernel barely responds (13) because there's no vertical edge at this location. Different kernel = different pattern detector, same underlying operation.

Now we've been doing this at individual positions. To build a complete "edge map," we slide the kernel across *every* valid position and collect the results into a new image. Each pixel in the output tells you "how strongly did the kernel's pattern match here?"

As we mentioned before, a **convolution** is the operation of applying a kernel over a grid this way — sliding, multiplying, and summing. The **feature map** is the result: the new matrix of all those output values collected together. One kernel produces one feature map. 16 kernels produce 16 feature maps — each one a "heat map" of where a different pattern was detected.

The paper ["A Guide to Convolution Arithmetic for Deep Learning"](https://arxiv.org/abs/1603.07285) has many great diagrams showing how kernels are applied. Here's an example showing (at the bottom) a light blue 4×4 image, with a dark blue 3×3 kernel, creating a 2×2 green output feature map at the top.

<img alt="Result of applying a 3×3 kernel to a 4×4 image" width="782" caption="Result of applying a 3×3 kernel to a 4×4 image (courtesy of Vincent Dumoulin and Francesco Visin)" id="three_ex_four_conv" src="images/att_00028.png">

Look at the shape of the result. If the original image has a height of `h` and a width of `w`, how many 3×3 windows can we find? As you can see from the example, there are `h-2` by `w-2` windows, so the image we get has a result as a height of `h-2` and a width of `w-2`.

We won't implement this convolution function from scratch, but use PyTorch's implementation instead (it is way faster than anything we could do in Python).

### Convolutions in PyTorch

Convolution is such an important operation that PyTorch has it built in as `F.conv2d` (where `F` is `torch.nn.functional`). It expects two main arguments:

- **input**: tensor of shape `(batch, in_channels, height, width)`
- **weight**: kernels of shape `(out_channels, in_channels, kernel_h, kernel_w)`

So both are rank-4 tensors. Our image and kernel are currently rank-2 (plain matrices), so we'll need to add dimensions. Why rank-4? Two reasons:

**Batching** — PyTorch can apply the convolution to every image in a batch simultaneously. That's the first axis.

**Multiple kernels** — PyTorch can apply many kernels at once, producing multiple feature maps in a single call. Let's create diagonal-edge kernels too and stack all four edge kernels into one tensor:

In [ ]:
# Two diagonal edge kernels
diag1_edge = tensor([[ 0,-1, 1],
                     [-1, 1, 0],
                     [ 1, 0, 0]]).float()
diag2_edge = tensor([[ 1,-1, 0],
                     [ 0, 1,-1],
                     [ 0, 0, 1]]).float()

# Stack all 4 kernels into one tensor: shape (4, 3, 3)
# 4 kernels, each 3x3 — we'll apply all of them at once using F.conv2d
edge_kernels = torch.stack([left_edge, top_edge, diag1_edge, diag2_edge])
edge_kernels.shape

To test this, we'll need a DataLoader and a sample mini-batch. Let's load MNIST and grab a batch:

In [ ]:
# Create a filtered dataset of just 3s and 7s for binary classification
class MNIST37(Dataset):
    """MNIST filtered to just digits 3 and 7, for binary classification."""
    def __init__(self, train=True):
        full = datasets.MNIST('./data', train=train, download=True,
                              transform=transforms.ToTensor())
        mask = (full.targets == 3) | (full.targets == 7)
        self.data = full.data[mask].unsqueeze(1).float() / 255.0  # (N, 1, 28, 28)
        self.targets = (full.targets[mask] == 7).long()  # 3->0, 7->1

    def __len__(self):
        return len(self.data)

    def __getitem__(self, i):
        return self.data[i], self.targets[i]

train_ds_37 = MNIST37(train=True)
valid_ds_37 = MNIST37(train=False)

train_dl_37 = DataLoader(train_ds_37, batch_size=64, shuffle=True)
valid_dl_37 = DataLoader(valid_ds_37, batch_size=64, shuffle=False)

# Grab one batch
xb, yb = next(iter(valid_dl_37))
xb.shape  # (64, 1, 28, 28)

One batch contains 64 images, each of 1 channel, with 28x28 pixels. `F.conv2d` can handle multichannel (i.e., color) images too. A *channel* is a single basic color in an image. For regular full-color images there are three channels: red, green, and blue. PyTorch represents an image as a rank-3 tensor, with dimensions `[channels, rows, columns]`.

Kernels passed to `F.conv2d` need to be rank-4 tensors: `[channels_out, channels_in, rows, columns]`. `edge_kernels` is currently missing one of these. We need to tell PyTorch that the number of input channels in the kernel is one, which we can do by inserting an axis of size one (this is known as a *unit axis*) in the first location, where the PyTorch docs show `in_channels` is expected. To insert a unit axis into a tensor, we use the `unsqueeze` method:

In [ ]:
# F.conv2d expects kernels shaped (out_channels, in_channels, H, W)
# Our kernels are (4, 3, 3) — missing the in_channels dimension
# unsqueeze(1) adds it: (4, 3, 3) → (4, 1, 3, 3) — 4 filters, 1 input channel each
edge_kernels.shape, edge_kernels.unsqueeze(1).shape

This is now the correct shape for `edge_kernels`. Let's pass this all to `conv2d`:

In [ ]:
# Reshape kernels to have the input channel dimension
edge_kernels = edge_kernels.unsqueeze(1)  # (4, 1, 3, 3)

In [ ]:
# Apply all 4 edge kernels to all 64 images in one shot
# Input: (64, 1, 28, 28) — 64 grayscale images
# Kernels: (4, 1, 3, 3) — 4 filters
# Output: (64, 4, 26, 26) — 64 images × 4 feature maps × 26x26 (shrunk by 2 due to no padding)
batch_features = F.conv2d(xb, edge_kernels)
batch_features.shape

The output shape shows we gave 64 images in the mini-batch, 4 kernels, and 26×26 edge maps (we started with 28×28 images, but lost one pixel from each side as discussed earlier). We can see we get the same results as when we did this manually:

In [ ]:
# Show the first feature map of the first image (left-edge detection)
show_image(batch_features[0,0])
plt.show()

The real power of PyTorch is that all of this — multiple kernels, multiple images, multiple channels — runs in parallel on the GPU. If we did these operations one at a time, we'd be hundreds of times slower. Using our manual Python loop from earlier, millions of times slower. Keeping the GPU busy with large batches of work is one of the key practical skills in deep learning.

It would be nice to not lose those two pixels on each axis. The way we do that is to add *padding*, which is simply additional pixels added around the outside of our image. Most commonly, pixels of zeros are added. 

### Strides and Padding

With appropriate padding, we can ensure that the output activation map is the same size as the original image, which can make things a lot simpler when we construct our architectures. The diagram below shows how adding padding allows us to apply the kernels in the image corners.

<img src="images/chapter9_padconv.svg" id="pad_conv" caption="A convolution with padding" alt="A convolution with padding" width="600">

With a 5×5 input, 4×4 kernel, and 2 pixels of padding, we end up with a 6×6 activation map.

<img alt="A 4×4 kernel with 5×5 input and 2 pixels of padding" width="783" caption="A 4×4 kernel with 5×5 input and 2 pixels of padding (courtesy of Vincent Dumoulin and Francesco Visin)" id="four_by_five_conv" src="images/att_00029.png">

If we use a kernel of size `ks` by `ks` (with `ks` an odd number), the necessary padding on each side to keep the same shape is `ks//2`. An even `ks` would require different padding on the top/bottom vs left/right, but in practice we almost always use odd kernel sizes.

So far, we've moved the kernel one pixel at a time. But we can jump further — for instance, moving two pixels after each application. This is called a *stride-2* convolution. The most common kernel size in practice is 3×3, and the most common padding is 1. Stride-2 convolutions are useful for decreasing the spatial size of our outputs, and stride-1 convolutions for adding depth without changing the output size.

<img alt="A 3×3 kernel with 5×5 input, stride-2 convolution, and 1 pixel of padding" width="774" caption="A 3×3 kernel with 5×5 input, stride-2 convolution, and 1 pixel of padding (courtesy of Vincent Dumoulin and Francesco Visin)" id="three_by_five_conv" src="images/att_00030.png">

In an image of size `h` by `w`, using a padding of 1 and a stride of 2 will give us a result of size `(h+1)//2` by `(w+1)//2`. The general formula for each dimension is `(n + 2*pad - ks)//stride + 1`, where `pad` is the padding, `ks`, the size of our kernel, and `stride` is the stride.

Now that we understand what a convolution is, let's use them to build a neural net.

## Our First Convolutional Neural Network

There's no reason to believe that hand-crafted edge kernels are the best possible kernels for image recognition. And in deeper layers, kernels need to detect much more complex patterns that we couldn't design manually.

Instead, we let the network learn its own kernel values through SGD. The model discovers whatever patterns are most useful for classification.

When we use convolutions instead of (or in addition to) regular linear layers, we create a *convolutional neural network* (CNN).

### Creating the CNN

Let's recall the simple MLP architecture from L9. It was defined like this:

In [ ]:
# For reference: the simple MLP from earlier chapters
# Flattens 28x28 image to 784 numbers, maps through one hidden layer
# This is what we're replacing with convolutions
simple_net = nn.Sequential(
    nn.Linear(28*28,30),  # 784 inputs → 30 hidden
    nn.ReLU(),
    nn.Linear(30,1)       # 30 → 1 output
)

We can view a model's definition:

In [ ]:
# Print the model structure
simple_net

We now want to create a similar architecture to this linear model, but using convolutional layers instead of linear. `nn.Conv2d` is the module equivalent of `F.conv2d`. It's more convenient than `F.conv2d` when creating an architecture, because it creates the weight matrix for us automatically when we instantiate it.

Here's a possible architecture:

In [ ]:
# Naive attempt: replace linear layers with conv layers
# Problem: output is still (batch, 1, 28, 28), not a single prediction
broken_cnn = nn.Sequential(
    nn.Conv2d(1, 30, kernel_size=3, padding=1),   # 1 channel in -> 30 channels out, same spatial size
    nn.ReLU(),
    nn.Conv2d(30, 1, kernel_size=3, padding=1)     # 30 -> 1 channel, still 28x28
)

One thing to note here is that we didn't need to specify 28×28 as the input size. That's because a linear layer needs a weight in the weight matrix for every pixel, so it needs to know how many pixels there are, but a convolution is applied over each pixel automatically. The weights only depend on the number of input and output channels and the kernel size, as we saw in the previous section.

Think about what the output shape is going to be, then let's try it and see:

In [ ]:
# Shape: (64, 1, 28, 28) — still a full image per sample, not a single class prediction!
# We need to shrink the spatial dimensions down to 1x1
broken_cnn(xb).shape

This is not something we can use to do classification, since we need a single output activation per image, not a 28×28 map of activations. One way to deal with this is to use enough stride-2 convolutions such that the final layer is size 1. That is, after one stride-2 convolution the size will be 14×14, after two it will be 7×7, then 4×4, 2×2, and finally size 1.

Let's try that now. First, we'll define a function with the basic parameters we'll use in each convolution:

In [ ]:
# Helper: create a conv layer with stride=2 (halves spatial dimensions each time)
# padding = kernel_size // 2 keeps the halving clean (ks=3 → padding=1, ks=5 → padding=2)
def conv(in_channels, out_channels, kernel_size=3, activation=True):
    layer = nn.Conv2d(
        in_channels=in_channels,
        out_channels=out_channels,
        kernel_size=kernel_size,
        stride=2,
        padding=kernel_size // 2
    )
    if activation:
        layer = nn.Sequential(layer, nn.ReLU())
    return layer

**Why wrap `conv` in a helper:** Every layer uses the same stride, padding formula, and optional ReLU. The helper means you only change `in_channels` and `out_channels` per layer — less room for mistakes, and it's immediately clear what varies between layers.

**Why double the channels when spatial size halves?** Stride 2 halves both width and height, so a 14×14 feature map becomes 7×7 — that's ÷4 total values per feature map. If we kept the same number of channels, the layer would carry way less information. Doubling the channels partially compensates, so capacity stays roughly balanced across layers. Not a strict rule, just a convention that works well.

**Terminology note:** In CNNs, "channels" and "features" mean the same thing — the number of feature maps at a given layer. "This layer has 32 features" = "this layer has 32 channels" = 32 feature maps. This is different from tabular ML where "features" means columns. The word is overloaded depending on context.

Here is how we can build a simple CNN:

In [ ]:
# Helper: create a conv layer with stride=2 (halves spatial dimensions each time)
# padding = kernel_size // 2 keeps the halving clean (ks=3 -> padding=1, ks=5 -> padding=2)
def conv(in_channels, out_channels, kernel_size=3, activation=True):
    layer = nn.Conv2d(
        in_channels=in_channels,
        out_channels=out_channels,
        kernel_size=kernel_size,
        stride=2,
        padding=kernel_size // 2
    )
    if activation:
        layer = nn.Sequential(layer, nn.ReLU())
    return layer

# Build the CNN: each conv layer halves spatial dimensions (stride=2) and grows channels
# Spatial:  28x28 -> 14x14 -> 7x7 -> 4x4 -> 2x2 -> 1x1
# Channels:   1  ->   4   ->  8  -> 16  -> 32  ->  2 (one per class)
simple_cnn = nn.Sequential(
    conv(in_channels=1,  out_channels=4),               # 28x28 -> 14x14
    conv(in_channels=4,  out_channels=8),               # 14x14 -> 7x7
    conv(in_channels=8,  out_channels=16),              # 7x7   -> 4x4
    conv(in_channels=16, out_channels=32),              # 4x4   -> 2x2
    conv(in_channels=32, out_channels=2, activation=False),  # 2x2 -> 1x1, no ReLU (raw logits)
    nn.Flatten(),                                       # (batch, 2, 1, 1) -> (batch, 2)
)

The comments after each conv layer show how the spatial dimensions change. They assume a 28×28 input — trace through the numbers to make sure you understand the halving pattern.

Now the network outputs two activations, which map to the two possible levels in our labels:

In [ ]:
# Verify: output is (64, 2) — 64 images, 2 class scores each
simple_cnn(xb).shape

Let's train this CNN and see how it does. We'll use the training loop helpers we defined earlier:

In [ ]:
# Train the simple CNN on 3s vs 7s
model = simple_cnn.to(device)
optimizer = optim.Adam(model.parameters(), lr=0.01)
scheduler = optim.lr_scheduler.OneCycleLR(optimizer, max_lr=0.01,
                                          steps_per_epoch=len(train_dl_37), epochs=10)

for epoch in range(10):
    train_loss, train_acc = train_one_epoch(model, train_dl_37, F.cross_entropy, optimizer, scheduler)
    val_loss, val_acc = evaluate(model, valid_dl_37, F.cross_entropy)
    if (epoch + 1) % 2 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:2d}  Train acc: {train_acc:.3f}  Val acc: {val_acc:.3f}")

To see the shapes flowing through the model, let's pass a dummy batch and print each layer's output:

In [ ]:
# Shape trace: watch dimensions shrink through the network
x = torch.randn(1, 1, 28, 28).to(device)
for i, layer in enumerate(model):
    x = layer(x)
    print(f"After layer {i}: {x.shape}")

# Total parameters
total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal parameters: {total_params:,}")

Note that the output of the final `Conv2d` layer is `1x2x1x1`. We need to remove those extra `1x1` axes; that's what `nn.Flatten()` does. It squeezes the spatial dimensions so we end up with a flat vector of class scores.

Let's see the training results above. Since this is a deeper network than we've built from scratch before, we used a moderate learning rate with 1cycle scheduling:

The model learns to distinguish 3s from 7s with high accuracy. We still have a few more tricks to learn, but we're getting closer and closer to being able to create a modern CNN from scratch.

### Understanding Convolution Arithmetic

We can see from the summary that we have an input of size `64x1x28x28`. The axes are `batch,channel,height,width`. This is often represented as `NCHW` (where `N` refers to batch size). Tensorflow, on the other hand, uses `NHWC` axis order. The first layer is:

In [ ]:
# Grab the first block of the model (Conv2d + ReLU)
m = model[0]
m

So we have 1 input channel, 4 output channels, and a 3×3 kernel. Let's check the weights of the first convolution:

In [ ]:
# Weight shape: (out_channels, in_channels, kernel_h, kernel_w) = (4, 1, 3, 3)
# 4 filters, each looking at 1 input channel, each 3x3
# Total weight params: 4 × 1 × 3 × 3 = 36
m[0].weight.shape

The summary shows we have 40 parameters, and `4*1*3*3` is 36. What are the other four parameters? Let's see what the bias contains:

In [ ]:
# Bias shape: (4,) — one bias per output channel
# 36 weights + 4 biases = 40 total parameters (matches the summary)
m[0].bias.shape

There is one bias per output channel (4 channels = 4 biases). The output shape is `64x4x14x14`, which becomes the input to the next layer. Let's trace the computation to understand why we double channels after each stride-2 conv.

The next layer has 296 parameters. Ignoring the batch axis and biases: for each of `14*14=196` spatial positions, we multiply `296-8=288` weights, so that's `196*288=56_448` multiplications. The next layer has `7*7*(1168-16)=56_448` multiplications — the same number.

The stride-2 conv halved the grid from `14x14` to `7x7`, and we doubled the channels from 8 to 16. The grid shrank by 4×, the channels grew by 2×, so computation stayed roughly constant. If we kept channels the same at each layer, computation would shrink as we go deeper. But deeper layers need to represent more complex patterns (edges combine into textures, textures into shapes, shapes into objects), so maintaining capacity makes sense.

Another way to think of this is based on receptive fields.

### Receptive Fields

The *receptive field* is the area of an image that is involved in the calculation of a layer. The spreadsheet `multi-channel-conv-example.xlsx` (in this lesson's folder) demonstrates the full CNN pipeline on a real MNIST digit with trace precedents. You can click on any cell in the conv2 section and trace back to see which input pixels contributed to that value.

The image below shows what happens when you click on one cell in the conv2 output and trace its precedents:

<img alt="Immediate precedents of conv2 layer" width="308" caption="Immediate precedents of Conv2 layer" id="preced1" src="images/att_00068.png">

The cell with the green border is the one we clicked on, and the blue highlighted cells are its *precedents* — the cells used to calculate its value. These are the corresponding 3×3 area from the input layer (on the left) and the kernel weights (on the right). Let's click *trace precedents* again to see what feeds into *those* inputs:

<img alt="Secondary precedents of conv2 layer" width="601" caption="Secondary precedents of Conv2 layer" id="preced2" src="images/att_00069.png">

We have two stride-2 conv layers, so tracing back from Conv2 reaches all the way to the input image. A 7×7 area of input pixels contributes to a single value in the Conv2 output — that 7×7 area is the *receptive field*. Each layer also has its own set of learned kernels, so two layers means two sets of weights.

The deeper you go (more stride-2 convs), the larger the receptive field. A large receptive field means each activation summarizes a bigger chunk of the input image. Deeper layers represent complex, high-level patterns (eyes, textures, shapes) that span many pixels, so they need both large receptive fields *and* more channels to capture that complexity. This is another way of seeing why we increase channels after each stride-2 conv.

## Color Images

So far we've worked with grayscale images (1 channel). But real-world images are usually color — three channels for red, green, and blue. In PyTorch, a color image is a rank-3 tensor with shape `(3, H, W)`:

In [ ]:
# Load a color image from CIFAR-10 (real photos, 32x32, 10 classes)
cifar = datasets.CIFAR10('./data', train=True, download=True, transform=transforms.ToTensor())

im, label = cifar[0]
print(f"Shape: {im.shape}  |  Label: {cifar.classes[label]}")

# Display (upscaled for visibility — the actual image is only 32x32)
plt.figure(figsize=(3, 3))
plt.imshow(im.permute(1, 2, 0))  # (C, H, W) -> (H, W, C) for matplotlib
plt.axis('off')
plt.title(cifar.classes[label])
plt.show()

The first axis contains the channels, red, green, and blue:

In [ ]:
# Split into R, G, B channels — each is a 32x32 grayscale "heat map"
fig, axes = plt.subplots(1, 3, figsize=(10, 3))
for channel, ax, cmap, name in zip(im, axes, ['Reds', 'Greens', 'Blues'], ['Red', 'Green', 'Blue']):
    ax.imshow(channel, cmap=cmap)
    ax.set_title(f'{name} channel')
    ax.axis('off')
plt.tight_layout()
plt.show()

So far we've convolved a single kernel over a single-channel (grayscale) image. But a color image has 3 input channels. How does convolution handle that?

The key idea: a kernel for a multi-channel input isn't 2D anymore — it's 3D. For an RGB image with a 3×3 kernel, the kernel shape is `3×3×3`: one 3×3 slice per input channel (red, green, blue), each with its own learned weights. At each spatial position, we multiply all three slices against the corresponding image patch, sum everything up, and get a single number.

We can have as many output channels as we want — each one is a separate 3D kernel that specializes in detecting a different pattern. The diagram below shows how one kernel operates on an RGB image:

<img src="images/chapter9_rgbconv.svg" id="rgbconv" caption="Convolution over an RGB image" alt="Convolution over an RGB image" width="550">

To apply a convolution to a color image, the kernel must have the same number of slices as the image has channels. At each position, the corresponding slices of the kernel and image are multiplied together, and all the products are summed into a single number per output position, as shown below.

The kernel is rank-3 (3×3×3 for an RGB image — one 3×3 slice per color channel). But the feature map it produces is a flat 2D grid. The channel dimension gets collapsed by summing: 9 products from the red slice + 9 from green + 9 from blue = 27 multiplications, all summed into one number per position.

<img src="images/chapter9_rgb_conv_stack.svg" id="rgbconv2" caption="Summing across RGB channel slices" alt="Summing across RGB channel slices" width="500">

With multiple output channels, we get one 3D kernel per output channel. The full weight tensor has shape `(out_channels, in_channels, kernel_h, kernel_w)` in PyTorch. Each output channel also has one bias value, so the biases form a vector of length `out_channels`.

There are no special mechanisms needed for color images — just make sure your first conv layer's `in_channels` matches the number of color channels (3 for RGB). You can also convert to grayscale or HSV, but experiments generally show that changing color encoding doesn't help as long as you don't lose information. Converting to grayscale *does* lose information and can hurt (a pet breed might have a distinctive color). Converting RGB to HSV generally makes no difference.

Those famous visualizations of "what a neural net learns" from the [Zeiler and Fergus paper](https://arxiv.org/abs/1311.2901) show exactly this — the three RGB slices of learned first-layer kernels, displayed as small color images:

<img alt="Layer 1 kernels found by Zeiler and Fergus" width="120" src="images/att_00031.png">

Each small image shows the three RGB slices of one learned kernel, displayed as a color image. Even though nobody told the network to look for edges, it discovered edge detectors (and color gradient detectors) on its own through SGD.

Now let's see how we can train these CNNs, and explore the techniques that make training stable and efficient.

## Improving Training Stability

Since we are so good at recognizing 3s from 7s, let's move on to something harder: recognizing all 10 digits. That means we'll need the full MNIST dataset:

In [ ]:
# Full MNIST: all 10 digits (0-9), 60K training + 10K test images
# Normalize with MNIST mean/std for stable training
def get_dls(bs=64):
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,))  # MNIST mean, std
    ])
    train_ds = datasets.MNIST('./data', train=True, download=True, transform=transform)
    valid_ds = datasets.MNIST('./data', train=False, download=True, transform=transform)
    train_dl = DataLoader(train_ds, batch_size=bs, shuffle=True)
    valid_dl = DataLoader(valid_ds, batch_size=bs, shuffle=False)
    return train_dl, valid_dl

train_dl, valid_dl = get_dls()

We've set up a `get_dls` function that makes it easy to change batch size later. The `Normalize` transform standardizes pixel values using MNIST's precomputed mean and standard deviation, giving our model a cleaner signal to learn from.

Remember, it's always a good idea to look at your data before you use it:

In [ ]:
# Sanity check: visualize a batch to make sure data looks right
xb_show, yb_show = next(iter(train_dl))
fig, axes = plt.subplots(3, 3, figsize=(4, 4))
for ax, img, label in zip(axes.flat, xb_show[:9], yb_show[:9]):
    show_image(img, ax=ax)
    ax.set_title(str(label.item()), fontsize=10)
plt.tight_layout()
plt.show()

Now that we have our data ready, we can train a simple model on it.

### A Simple Baseline

Earlier in this lesson, we built a model based on a `conv` function like this:

In [ ]:
# Same conv helper as before: stride-2 convolution + optional ReLU
def conv(in_channels, out_channels, kernel_size=3, activation=True):
    layer = nn.Conv2d(
        in_channels=in_channels,
        out_channels=out_channels,
        kernel_size=kernel_size,
        stride=2,
        padding=kernel_size // 2
    )
    if activation:
        layer = nn.Sequential(layer, nn.ReLU())
    return layer

We're switching from 2-class (3 vs 7) to 10-class (all digits), so we need more channels throughout the network. We double the first layer from 4 to 8, which doubles every layer after it.

The first layer also uses a 5×5 kernel instead of 3×3. A 5×5 kernel sees a 25-pixel neighborhood per channel vs only 9 for a 3×3, giving the first layer more context to work with when all it has is raw pixel values. Later layers stick with 3×3 because their inputs are already processed feature maps with richer information per pixel.

In [ ]:
# 10-class CNN: more channels than the 2-class version, 10 output channels (one per digit)
# First layer uses kernel_size=5 for a larger receptive field from the start
def simple_cnn():
    return nn.Sequential(
        conv(in_channels=1,  out_channels=8,  kernel_size=5),      # 28x28 -> 14x14
        conv(in_channels=8,  out_channels=16),                     # 14x14 -> 7x7
        conv(in_channels=16, out_channels=32),                     # 7x7   -> 4x4
        conv(in_channels=32, out_channels=64),                     # 4x4   -> 2x2
        conv(in_channels=64, out_channels=10, activation=False),   # 2x2   -> 1x1, 10 classes
        nn.Flatten(),
    )

Our CNN works for 3s vs 7s, but it will struggle with all 10 digits. To understand why, we'll track what's happening inside the network during training using our `ActivationTracker`. It records the mean, standard deviation, and distribution of activation values at every layer, so we can see whether information is actually flowing through the network or dying off.

Let's train at a learning rate of 0.15 and see what happens:

In [ ]:
# Training helper with activation tracking
# Returns (model, tracker) so we can inspect layer stats after training
def fit(epochs=1, lr=0.15, one_cycle=False, bs=None):
    if bs:
        global train_dl, valid_dl
        train_dl, valid_dl = get_dls(bs)

    model = simple_cnn().to(device)
    tracker = ActivationTracker(model)
    optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9)
    scheduler = None
    if one_cycle:
        scheduler = optim.lr_scheduler.OneCycleLR(
            optimizer, max_lr=lr,
            steps_per_epoch=len(train_dl), epochs=epochs)

    lrs = []  # track learning rates for plotting
    for epoch in range(epochs):
        model.train()
        for xb, yb in train_dl:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            out = model(xb)
            loss = F.cross_entropy(out, yb)
            loss.backward()
            optimizer.step()
            lrs.append(optimizer.param_groups[0]['lr'])
            if scheduler:
                scheduler.step()

        val_loss, val_acc = evaluate(model, valid_dl, F.cross_entropy)
        print(f"Epoch {epoch+1}  Val acc: {val_acc:.3f}")

    tracker.remove_hooks()
    return model, tracker, lrs

In [ ]:
# Train 1 epoch at lr=0.15. This will train poorly, and we'll diagnose why
model, tracker, lrs = fit()

The model barely learned. A simple MLP can hit 97%+ on MNIST, so something is clearly wrong. Let's use our `ActivationTracker` to look inside the network and find out what's happening:

In [ ]:
# Plot activation stats for the FIRST layer (index 0)
# Shows mean, std, and % of near-zero activations over training batches
tracker.plot_layer_stats(0)

The plot shows three metrics across training batches (each batch = one forward pass on a chunk of images):

**Mean (left)** — The average activation value at this layer. It's hovering slightly below zero and a bit noisy, but roughly stable. Not ideal, but not catastrophic — the first layer is receiving raw pixel data directly, so it has a decent signal to work with.

**Std (middle)** — How spread out the activation values are. It's bouncing around but staying in a reasonable range. The layer is producing a variety of output values, which means it can still distinguish between different input patches.

**% near zero (right)** — What fraction of activations are effectively dead (near zero after ReLU). It's around 50% — not great, but the layer is still passing some signal through.

The first layer looks okay. But problems compound as you go deeper — each layer feeds into the next, so small issues amplify. Let's check the penultimate layer (second-to-last, right before the final prediction):

In [ ]:
# Plot activation stats for the PENULTIMATE layer
# Problems compound deeper in the network, so this layer usually shows the worst issues
tracker.plot_layer_stats(-2)

The x-axis on all three charts is **training batches** — each tick is one batch of images passed through the network.

**Mean (left)** — The average activation value at this layer. If it's deeply negative and erratic, that's bad: most values before ReLU are negative, so ReLU kills them. A healthy mean would be near zero and smooth.

**Std (middle)** — How spread out the activation values are. If it collapses over time, the activations are shrinking — the layer is losing its ability to distinguish between different inputs. A healthy std stays stable around 1-2.

**% near zero (right)** — The killer. When a high percentage of activations at this layer are near zero, the layer is effectively dead — outputting zeros for nearly every input. The final prediction layer has almost nothing to work with.

The story these three tell together: if the mean is negative, ReLU kills most values, nearly everything is zero, and the network can barely learn. That's why our accuracy is so low despite this being "easy" MNIST.

### Increase Batch Size

One possible cause: noisy gradients. With a batch size of 64, each gradient is estimated from only 64 images. Individual noisy examples can push the weights in bad directions. Larger batches average over more samples, giving smoother and more reliable gradient estimates. The tradeoff: fewer batches per epoch means fewer weight updates. Let's try batch size 512:

In [ ]:
# Experiment 1: increase batch size from 64 -> 512
# Larger batches = smoother gradient estimates = more stable training
train_dl, valid_dl = get_dls(512)

In [ ]:
# Retrain with larger batch size
model, tracker, lrs = fit()

Still poor. With 512 samples per batch, we only get ~117 gradient updates per epoch instead of ~938. The model sees less variety per pass. But the real issue isn't batch size — let's check whether the activations improved:

In [ ]:
# Check penultimate layer. Still lots of near-zero activations?
tracker.plot_layer_stats(-2)

The activations are still dying. Smoother gradients from the larger batch didn't fix the underlying problem — activations still drift toward zero through the deeper layers. We need something that addresses the training dynamics directly, not just the gradient quality.

### 1cycle Training

One more thing to try before we look at the real solution. The problem could be lr-related: starting with a high learning rate when weights are still random can destabilize training immediately — the updates are too aggressive for a network that hasn't learned anything yet. The **1cycle policy** (a learning rate schedule, not an optimizer) addresses this: start with a low lr (safe warm-up while weights are random), gradually ramp up to a high lr (fast learning once the network has found a reasonable direction), then decay back down (fine-tune the details at the end).

In [ ]:
# Experiment 2: use 1cycle learning rate schedule
# 1cycle: lr starts low, ramps up to max, then decays
# This avoids destabilizing the model early when weights are random

In [ ]:
# Train with 1cycle schedule
model, tracker, lrs = fit(one_cycle=True)

Still struggling. Without healthy activations, even a well-designed lr schedule can't save the model — the signal through the deeper layers is already dead by the time the gradients arrive. Let's see the lr curve and check the activations:

In [ ]:
# Visualize the learning rate schedule: warmup then cosine annealing
plt.plot(lrs)
plt.xlabel('Batch')
plt.ylabel('Learning Rate')
plt.title('1cycle Learning Rate Schedule')
plt.show()

PyTorch's `OneCycleLR` uses cosine annealing for the decay phase, a smooth curve instead of linear decay, which tends to work slightly better in practice. You can see the warm-up and decay shape in the plot above.

Let's check if the activations improved:

In [ ]:
# Penultimate layer stats with 1cycle. Should be more stable
tracker.plot_layer_stats(-2)

The activations are still dead. We've now tried two common training improvements — larger batches and learning rate scheduling — and neither fixed the root cause. Both techniques address gradient-related problems (noisy updates, lr too high/low), but our problem is deeper: the activations themselves are collapsing to zero as they flow through the network. We need something that directly stabilizes the activation distributions.

### Batch Normalization

Let's recap. We've tried two common training improvements — larger batches and 1cycle — and neither fixed the root cause. The `plot_layer_stats` keep showing the same thing: most activations at the deeper layers are near zero. The network can't learn effectively *despite* our best efforts at tuning batch size and learning rate schedules.

Why does this happen? Each layer takes the previous layer's output (activations) and transforms it. If one layer's output drifts slightly toward zero, the next layer receives smaller inputs, which makes *its* output drift further toward zero. This compounds — by layer 5, the signal from the original image is gone. This is called **activation drift**.

**Batch normalization** fixes this directly. After each conv layer, it normalizes the activations to have zero mean and unit variance — using the mean and std of the current training batch. Think of it as a reset button: no matter how much the previous layer's output drifted, batchnorm pulls it back to a healthy distribution before passing it on.

The simulation below shows this concept in isolation — no ReLU, just the drift and the reset:

The contrast is stark. Without batchnorm, the distribution collapses into a narrow spike — most values are near zero by layer 4. With batchnorm, the distribution stays spread out at every layer because the normalization step resets it after each transformation.

In [ ]:
# Simulating activation drift: what happens to values as they pass through layers

np.random.seed(42)

# Without batchnorm: activations drift and collapse
no_bn_layers = []
values = np.random.randn(10000)
for _ in range(4):
    no_bn_layers.append(values.copy())
    values = values * 0.5 - 0.3 + np.random.randn(10000) * 0.1  # shrink + shift

# With batchnorm: same drift, but reset after each layer
bn_layers = []
values = np.random.randn(10000)
for _ in range(4):
    bn_layers.append(values.copy())
    values = values * 0.5 - 0.3 + np.random.randn(10000) * 0.1  # same drift
    values = (values - values.mean()) / (values.std() + 1e-5)    # batchnorm reset

plot_activation_drift([
    {'values_per_layer': no_bn_layers, 'label': 'Without\nBatchNorm', 'color': 'steelblue'},
    {'values_per_layer': bn_layers,    'label': 'With\nBatchNorm',    'color': 'seagreen'},
])

The simulation shows the pure concept: without batchnorm, the distribution collapses into a narrow spike. With batchnorm, it stays spread out at every layer. But our real CNN also applies **ReLU** after each convolution, which changes the picture — ReLU zeros out all negative values, so the output of each layer is always ≥ 0.

Let's see what real MNIST activations look like. We'll pass actual images through two untrained models (random weights) — one without batchnorm and one with (using our notebook's Conv → ReLU → BN ordering):

In [ ]:
# Pass real MNIST images through two untrained models and compare activations

xb_sample, _ = next(iter(train_dl))  # grab a real batch of MNIST digits
xb_sample = xb_sample.to(device)

def get_layer_activations(model, x):
    """Forward pass through model, collect the output of each conv block."""
    activations = []
    hooks = []
    for layer in model:
        if isinstance(layer, nn.Sequential):  # each conv() block is an nn.Sequential
            def hook(module, input, output, store=activations):
                store.append(output.detach().cpu().flatten().numpy())
            hooks.append(layer.register_forward_hook(hook))
    with torch.no_grad():
        model(x)
    for h in hooks:
        h.remove()
    return activations

# Model WITHOUT batchnorm (uses current conv definition)
model_no_bn = simple_cnn().to(device)

# Model WITH batchnorm (ReLU then BN ordering, defined inline)
def conv_bn(in_channels, out_channels, kernel_size=3, activation=True):
    layers = [nn.Conv2d(in_channels=in_channels, out_channels=out_channels,
                        kernel_size=kernel_size, stride=2, padding=kernel_size // 2)]
    if activation:
        layers.append(nn.ReLU())
    layers.append(nn.BatchNorm2d(out_channels))
    return nn.Sequential(*layers)

model_bn = nn.Sequential(
    conv_bn(1, 8, 5), conv_bn(8, 16), conv_bn(16, 32),
    conv_bn(32, 64), conv_bn(64, 10, activation=False), nn.Flatten()
).to(device)

# Collect activations from each conv block
acts_no_bn = get_layer_activations(model_no_bn, xb_sample)
acts_bn = get_layer_activations(model_bn, xb_sample)

plot_activation_drift([
    {'values_per_layer': acts_no_bn, 'label': 'Without\nBatchNorm', 'color': 'steelblue'},
    {'values_per_layer': acts_bn,    'label': 'With\nBatchNorm',    'color': 'seagreen'},
], title='Real MNIST activations through layers (untrained, random weights)')

The top row shows activations without batchnorm: distributions narrow with depth and more neurons die. This matches the `plot_layer_stats` we saw earlier. The bottom row (with BN) keeps activations alive — the distributions stay spread out.

But notice the bottom row looks skewed, not the clean bell curve from our simulation. That's because of **layer ordering**. The BN model above uses **Conv → ReLU → BN**. ReLU kills all negatives first, so BN receives a lopsided (all ≥ 0) input. It re-centers around zero (which is why some values go negative), but the shape stays skewed.

The more common ordering in modern architectures (PyTorch's ResNet, the original BN paper) is **Conv → BN → ReLU**. Here, BN normalizes the full symmetric output from Conv into a clean bell curve, and *then* ReLU clips the negative half. Let's compare all three:

In [ ]:
# Same comparison but with BN BEFORE ReLU: Conv -> BN -> ReLU
def conv_bn_relu(in_channels, out_channels, kernel_size=3, activation=True):
    layers = [nn.Conv2d(in_channels=in_channels, out_channels=out_channels,
                        kernel_size=kernel_size, stride=2, padding=kernel_size // 2)]
    if activation:
        layers.append(nn.BatchNorm2d(out_channels))  # BN first
        layers.append(nn.ReLU())                      # then ReLU
    return nn.Sequential(*layers)

model_bn_relu = nn.Sequential(
    conv_bn_relu(1, 8, 5), conv_bn_relu(8, 16), conv_bn_relu(16, 32),
    conv_bn_relu(32, 64), conv_bn_relu(64, 10, activation=False), nn.Flatten()
).to(device)

acts_bn_relu = get_layer_activations(model_bn_relu, xb_sample)

plot_activation_drift([
    {'values_per_layer': acts_no_bn,   'label': 'No\nBatchNorm',    'color': 'steelblue'},
    {'values_per_layer': acts_bn,      'label': 'ReLU then BN',     'color': 'seagreen'},
    {'values_per_layer': acts_bn_relu, 'label': 'BN then ReLU\n(standard)', 'color': 'mediumorchid'},
], title='Real MNIST activations: effect of BatchNorm placement')

Three things to notice:

**Top row (no BN)**: Activations narrow and die off with depth. The distributions collapse and % dead increases. This is the activation drift we've been diagnosing.

**Middle row (ReLU then BN)**: Distributions stay alive but look skewed. BN is normalizing a post-ReLU distribution (all >= 0), so it can re-center but not remove the skew. Still, the signal survives through all layers.

**Bottom row (BN then ReLU, standard)**: The ~50% dead neurons look alarming but are actually **expected and healthy**. BN produces a symmetric distribution centered at zero, then ReLU clips the negative half. That's exactly 50%. This is ReLU's normal sparsity, not pathological dying neurons. The non-zero half forms a clean half-normal shape that stays consistent across layers.

Both orderings work well in practice. We'll use **Conv -> BN -> ReLU** (the standard ordering from the original BN paper and PyTorch's built-in ResNet) for the rest of this lesson.

In [ ]:
# Final conv helper: Conv -> BN -> ReLU (the standard ordering)
# BatchNorm normalizes activations (zero mean, unit variance) within each batch
# This prevents the activation distribution from drifting during training
def conv(in_channels, out_channels, kernel_size=3, activation=True):
    layers = [nn.Conv2d(
        in_channels=in_channels,
        out_channels=out_channels,
        kernel_size=kernel_size,
        stride=2,
        padding=kernel_size // 2
    )]
    if activation:
        layers.append(nn.BatchNorm2d(out_channels))  # normalize first
        layers.append(nn.ReLU())                       # then activate
    return nn.Sequential(*layers)

and fit our model:

In [ ]:
# Train with batchnorm. Should see a big improvement in accuracy
model, tracker, lrs = fit(one_cycle=True)

98%+ in just 1 epoch — a massive jump. None of our previous attempts came close to matching even a simple MLP on MNIST. Batchnorm is the fix that addresses the root cause: keeping activations healthy. Let's verify with `plot_layer_stats`:

In [ ]:
# Penultimate layer stats with batchnorm
tracker.plot_layer_stats(-2)

Compare this to our earlier penultimate layer plots:

- **Mean** — stable and near zero (was erratic and deeply negative before)
- **Std** — holding steady (was collapsing toward zero)
- **% near zero** — dramatically lower (was 95%+, now the neurons are actually alive)

The activations are healthy from the very first batch. Batchnorm fixed the root cause we've been chasing this whole section — activation drift. It's so effective that nearly all modern neural networks use it or a close variant (LayerNorm, GroupNorm).

And because the network is now stable, we can combine it with 1cycle and push to a higher learning rate. Let's train for 5 epochs at lr=0.1:

In [ ]:
# Now train for 5 epochs with a higher learning rate
# BatchNorm stabilizes training enough that we can push lr to 0.1
model, tracker, lrs = fit(5, lr=0.1, one_cycle=True)

At this point, we know how to recognize digits with high accuracy! It's time to move on to something harder: real photographs, variable sizes, and many more classes. That's the CNN project notebook, where we apply everything from this lesson to Oxford Pets.

## Conclusions

We've seen that convolutions are just a type of matrix multiplication, with two constraints on the weight matrix: some elements are always zero, and some elements are tied (forced to always have the same value). These constraints enforce a specific pattern of connectivity between layers.

These constraints allow us to use far fewer parameters in our model, without sacrificing the ability to represent complex visual features. That means we can train deeper models faster, with less overfitting. Although the universal approximation theorem shows that it should be *possible* to represent anything in a fully connected network in one hidden layer, we've seen now that in *practice* we can train much better models by being thoughtful about network architecture.

Convolutions are by far the most common pattern of connectivity we see in neural nets (along with regular linear layers, which we refer to as *fully connected*), but it's likely that many more will be discovered.

We've also seen how to interpret the activations of layers in the network to see whether training is going well or not, and how batchnorm helps stabilize training. In the next lesson, we'll apply these concepts to real-world images and explore fine-tuning pretrained models.

## Questionnaire

**Convolutions and kernels**

1. What is a convolution? Describe the operation step by step.
1. What is the difference between a kernel and a feature map?
1. Write out a 3×3 kernel matrix that detects top edges. What would a feature map from this kernel look like on an image with a horizontal line?
1. What is the value of any kernel applied to a 3×3 matrix of zeros? Why?
1. What is "padding"? What does `padding=1` do with a 3×3 kernel?
1. What is "stride"? Why does stride 2 replace max pooling in modern architectures?
1. What are the shapes of the `input` and `weight` parameters to PyTorch's `F.conv2d`?
1. What does "NCHW" mean?

**Channels and architecture**

1. What is a "channel"? How does the meaning change between the input image and later layers?
1. The word "feature" means different things in tabular ML vs CNNs. Explain both meanings.
1. A conv layer receives 4 input channels. What is the shape of a single kernel in that layer? How many numbers does it output per spatial position?
1. Why do we double the number of channels after each stride-2 conv?
1. Why does `simple_cnn` use a larger kernel (5×5) in the first layer?
1. What is a "receptive field"? What is the receptive field size after two stride-2 convolutions with 3×3 kernels?
1. What does `Flatten` do in the CNN? Where does it go and why?
1. How is a color (RGB) image represented as a tensor? How does a convolution work on it — what shape is the kernel?

**Training stability and diagnostics**

1. What is the difference between *weights* and *activations*?
1. What are the three statistics plotted by `plot_layer_stats`? What does the x-axis represent?
1. Why are activations near zero problematic? What causes this in deeper layers?
1. What is "activation drift" and why does it compound through layers?
1. What are the upsides and downsides of training with a larger batch size?
1. Why should we avoid using a high learning rate at the start of training?
1. What is 1cycle training? Describe the three phases and why each matters. Is it an optimizer or a scheduler?
1. What trainable parameters does a batch normalization layer contain? Why are they needed?
1. What statistics does batch normalization use during training? How about during validation/inference?
1. Why does batch normalization allow us to use a higher learning rate?

### Further Research

1. What features other than edge detectors have been used in computer vision (especially before deep learning became popular)?
1. There are other normalization layers available in PyTorch (LayerNorm, GroupNorm, InstanceNorm). Try them out and see what works best. When would you use each?
1. Try moving the activation function (ReLU) before the batch normalization layer in `conv`. Does it make a difference? Research what order is recommended and why.
1. Open `multi-channel-conv-example.xlsx` and experiment with trace precedents to see how receptive fields grow through layers.

<div style="text-align: center; color: #888; font-size: 0.85em; margin-top: 40px; padding-top: 10px; border-top: 1px solid #ddd;">
© 2025 Utvecklarakademin UA Aktiebolag. All rights reserved.<br>
This material is proprietary and may not be reproduced, distributed, or shared without written permission.
</div>